In [ ]:
import pandas as pd

orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')


orders.head()

In [6]:
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [8]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [ ]:
# Convertendo colunas para datetime

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])

orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])

orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])


# Tempo de entrega

orders['delivery_time'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_purchase_timestamp']
)

orders[['order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_time']].head()




,order_purchase_timestamp,order_delivered_customer_date,delivery_time
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8 days 10:28:40
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13 days 18:46:08
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9 days 09:27:40
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13 days 05:00:36
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2 days 20:58:23


In [25]:
#calcula a diefrenca entrega

orders['delay'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_estimated_delivery_date']
)

#Tranorma a diferenca em dias

orders['delay_days'] = orders['delay'].dt.days

#Elimina linhas que não tem informação de atraso na entrega
orders = orders.dropna(subset=['delay_days'])

# cria coluna com status de entrega
orders['delivery_status'] = orders['delay_days'].apply(lambda x: 'Atrasado' if x > 0 else 'No prazo')

orders['delivery_status'].value_counts()


delivery_status
No prazo    89941
Atrasado     6535
Name: count, dtype: int64

In [31]:
order_items_full = items.merge(orders, on='order_id', how='inner')

order_items_full['revenue'] = order_items_full['price']

order_items_full['revenue'].sum()

order_items_full['month'] = order_items_full['order_purchase_timestamp'].dt.to_period('M')

order_items_full.groupby('month')['revenue'].sum()


#Realizei o join entre pedidos e itens para consolidar os dados, calculei a receita, agreguei por mês e obtive o faturamento mensal

month
2016-09       134.97
2016-10     40941.30
2016-12        10.90
2017-01    111798.36
2017-02    234223.40
2017-03    359198.85
2017-04    340669.68
2017-05    489159.25
2017-06    421923.37
2017-07    481604.52
2017-08    554699.70
2017-09    607399.67
2017-10    648247.65
2017-11    987648.07
2017-12    726033.19
2018-01    924645.00
2018-02    826467.12
2018-03    953356.25
2018-04    973534.09
2018-05    977544.69
2018-06    855591.97
2018-07    867486.47
2018-08    838576.64
Freq: M, Name: revenue, dtype: float64

In [32]:
order_items_full['revenue'].sum()
 

#A análise indica que o faturamento total no período foi de aproximadamente R$ 13,22 milhões, considerando os valores dos produtos vendidos.

np.float64(13220895.11)

In [35]:
#Ranking de produtos que mais faturam

order_items_full = order_items_full.merge(
    products,
    on='product_id',
    how='inner'
)

top_produtos = (
     order_items_full
    .groupby('product_category_name')['revenue']
    .sum()
    .sort_values(ascending=False)
)

top_produtos.head(10)

product_category_name
beleza_saude              1233211.61
relogios_presentes        1165898.98
cama_mesa_banho           1023434.76
esporte_lazer              954695.05
informatica_acessorios     888613.62
moveis_decoracao           711927.69
utilidades_domesticas      615628.69
cool_stuff                 610204.10
automotivo                 578849.35
brinquedos                 471197.48
Name: revenue, dtype: float64

In [ ]:
#Ranking dos 10 produtos mais vendidos

top_qtd = (
    order_items_full
    .groupby('product_category_name')['order_id']
    .count()
    .sort_values(ascending=False)
)

top_qtd.head(10)

product_category_name
cama_mesa_banho           10953
beleza_saude               9467
esporte_lazer              8431
moveis_decoracao           8160
informatica_acessorios     7643
utilidades_domesticas      6795
relogios_presentes         5857
telefonia                  4430
ferramentas_jardim         4268
automotivo                 4139
Name: order_id, dtype: int64

In [51]:
faturamento_mensal = (
    order_items_full
    .groupby('month')['revenue']
    .sum()
    .reset_index()
)



top_month_vendas = (
     order_items_full
    .groupby('month')[ 'revenue']
    .count()
    .sort_values(ascending=False)
)

top_month_vendas

month
2017-11    8474
2018-01    8037
2018-03    8017
2018-04    7827
2018-05    7810
2018-02    7519
2018-08    7142
2018-06    7007
2018-07    6960
2017-12    6187
2017-10    5214
2017-08    4797
2017-09    4737
2017-07    4416
2017-05    4003
2017-06    3489
2017-03    2897
2017-04    2569
2017-02    1858
2017-01     913
2016-10     319
2016-09       3
2016-12       1
Freq: M, Name: revenue, dtype: int64

In [43]:
faturamento_mensal = (
    order_items_full
    .groupby('month')['revenue']
    .sum()
    .sort_values()
)

crescimento = faturamento_mensal.pct_change() * 100

crescimento




month
2016-12             NaN
2016-09     1138.256881
2016-10    30233.629695
2017-01      173.069883
2017-02      109.505220
2017-04       45.446475
2017-03        5.439043
2017-06       17.462339
2017-07       14.145021
2017-05        1.568658
2017-08       13.398592
2017-09        9.500631
2017-10        6.725058
2017-12       11.999355
2018-02       13.833242
2018-08        1.465215
2018-06        2.029073
2018-07        1.390207
2018-01        6.588982
2018-03        3.105111
2018-04        2.116506
2018-05        0.411963
2017-11        1.033547
Freq: M, Name: revenue, dtype: float64

In [44]:
crescimento.sort_values(ascending=False).head(1)

month
2016-10    30233.629695
Freq: M, Name: revenue, dtype: float64

In [ ]:
# faturamento por mês (ordenado corretamente)
faturamento_mensal = (
    order_items_full
    .groupby('month')['revenue']
    .sum()
    .sort_index()
)

# crescimento percentual mês a mês
crescimento = faturamento_mensal.pct_change()

# pegar o maior crescimento
mes_maior = crescimento.idxmax()
valor_maior = crescimento.max()




In [48]:
print(f"O mês com maior crescimento foi {mes_maior} com aumento de {valor_maior:.2%}")

O mês com maior crescimento foi 2017-01 com aumento de 1025573.03%


In [52]:
order_items_full.to_csv('dados_tratados.csv', index=False)